In [0]:
# get the table name from widget (or use default)
dbutils.widgets.text("Database", "sebraepe_dev")
database =  dbutils.widgets.get("Database")

if database not in ["sebraepe_dev", "sebraepe_prod"]:
    raise ValueError("O valor do widget Database deve ser 'sebraepe_dev' ou 'sebraepe_prod' e não pode ser nulo.")

In [0]:
%run /Workspace/sebrae_pe/common/utils/environment

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window as W
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql.types import *

In [0]:
# read table
data = spark.read.option("mergeSchema", "true").parquet(
    "/Volumes/sebraepe_dev/landing/raw/tests/testes_sample.parquet")

In [0]:
# converte as strings para timestamp
data = data.withColumn("dthr_internacao", F.to_timestamp("dthr_internacao")) \
       .withColumn("dthr_alta_medica", F.to_timestamp("dthr_alta_medica"))

# calcula a diferenca em dias (alta - internação)
data = data.withColumn("dias_internado",
                   F.round(F.datediff("dthr_alta_medica", "dthr_internacao"), 2))

🔹 Ambiente definido: dev
🔹 Catálogo ativo: sebraepe_dev


In [0]:
# filter cases above august 2022
data = data.filter(F.col("dthr_alta_medica") >= F.lit("2022-08-01"))

In [0]:
# calcula a diferenca em dias (alta - internação)
data = data.withColumn("dias_internado",
                   F.round(F.datediff("dthr_alta_medica", "dthr_internacao"), 2))

# cid agg
data = data.withColumn("cid_agg", F.substring(F.col("cid"), 1, 3))


In [0]:
data.printSchema()

root
 |-- atendimento: long (nullable = true)
 |-- cid: string (nullable = true)
 |-- prontuario: double (nullable = true)
 |-- dthr_internacao: timestamp (nullable = true)
 |-- dthr_alta_medica: timestamp (nullable = true)
 |-- dt_saida_paciente: string (nullable = true)
 |-- descricao_anamnese: string (nullable = true)
 |-- descricao_nota_adicional_anamnese: string (nullable = true)
 |-- descricao_origem_evento: string (nullable = true)
 |-- flag_obito_internacao: string (nullable = true)
 |-- dt_obito: string (nullable = true)
 |-- descricao_tipo_alta_medica: string (nullable = true)
 |-- dias_internado: integer (nullable = true)
 |-- cid_agg: string (nullable = true)



In [0]:
# ---- parâmetros ----
min_support = 40        # mínimo de registros por CID para entrar no ranking (ajuste)
high_pct = 0.80          # corte para "alta permanência" (P80 por padrão)

# ---- sanity: tipagem e nulos ----
data1 = (
    data.select("cid_agg" , F.col("dias_internado").cast("double").alias("dias_internado"))
      .filter(F.col("cid_agg").isNotNull() & F.col("dias_internado").isNotNull())
)

In [0]:
# ---- 1) define "alta permanência" pelo percentil global (ex.: P80) ----
p80 = data1.select(F.expr(f"percentile_approx(dias_internado, {high_pct})").alias("p")).collect()[0]["p"]
data_flag = data1.withColumn("is_high_stay", (F.col("dias_internado") >= F.lit(p80)).cast("int"))
p80

13.0

In [0]:
# ---- 2) taxa global de alta permanência ----
p_global = data_flag.agg(F.avg("is_high_stay").alias("p")).collect()[0]["p"]
p_global

0.21242774566473988

In [0]:
# ---- 3) métricas por CID: suporte, taxa de alta, lift, médias úteis ----
by_cid = (
    data_flag.groupBy("cid_agg")
           .agg(
               F.count("*").alias("n"),
               F.avg("is_high_stay").alias("rate_high"),
               F.avg("dias_internado").alias("mean_days"),
               F.expr("percentile_approx(dias_internado, 0.5)").alias("p50_days"),
               F.expr("percentile_approx(dias_internado, 0.9)").alias("p90_days"),
           )
           .withColumn("lift_high", F.col("rate_high") / F.lit(p_global))
)
display(by_cid)

cid_agg,n,rate_high,mean_days,p50_days,p90_days,lift_high
Z21,1,0.0,1.0,1.0,1.0,0.0
M19,1,0.0,4.0,4.0,4.0,0.0
I31,2,1.0,19.0,19.0,19.0,4.707482993197279
Q61,3,0.0,5.666666666666667,6.0,10.0,0.0
O12,4,0.0,2.5,3.0,3.0,0.0
R16,4,0.5,49.0,9.0,91.0,2.3537414965986394
F25,1,1.0,16.0,16.0,16.0,4.707482993197279
Z12,2,0.0,7.0,7.0,7.0,0.0
M54,8,0.25,16.625,10.0,40.0,1.1768707482993197
D81,1,0.0,1.0,1.0,1.0,0.0


In [0]:

# ---- 4) filtra por suporte e pega Top-50 por lift (ou troque para mean_days se preferir) ----
top50_data = (
    by_cid.filter(F.col("n") >= F.lit(min_support))
          .orderBy(F.col("lift_high").desc(), F.col("p90_days").desc())
          .limit(50)
          .select("cid_agg")
)

top50 = [r[0] for r in top50_data.collect()]

display(top50_data)


cid_agg
P07
I73
I50
P22
M32
C18
I70
E10
C16
Z94


In [0]:

# ---- 5) coluna final com "Outros" para quem não está no Top-50 ----
data = data.withColumn(
    "cid_agg_top50",
    F.when(F.col("cid_agg" ).isin(top50), F.col("cid_agg" )).otherwise(F.lit("Outros"))
)

In [0]:
def generate_inter_features(data):

    # groupby average value by prontuario + cid_agg_top50
    w_prompt_param = W.partitionBy("prontuario", "cid_agg_top50")

    agg_mean =( 
               (data.withColumn("qtd_media_dias_internado", F.mean("dias_internado").over(w_prompt_param))
                .select(
                    "prontuario",
                    "cid_agg_top50",
                    "qtd_media_dias_internado"
                    )
        ).groupBy("prontuario")
        .pivot("cid_agg_top50")
        .agg(F.first("qtd_media_dias_internado"))
    )

    
    # renomeamos as colunas para adicionar os sufixos
    for col in agg_mean.columns[1:]:
        agg_mean = agg_mean.withColumnRenamed(col, f"{col.lower()}_qtd_media_dias_internado")

    # groupby max date value by prontuario + param_final
    w_orderdate = W.partitionBy("prontuario", "cid_agg_top50").orderBy(F.col("dthr_alta_medica").desc())

    agg_max_date = (
                (data.withColumn("rn", F.row_number().over(w_orderdate))
                .filter(F.col("rn") == 1)
                .select(
                   "prontuario",
                    "cid_agg_top50",
                    "dias_internado"
                )
        ).groupBy("prontuario")
        .pivot("cid_agg_top50")
        .agg(F.first("dias_internado"))
    )

    for col in agg_max_date.columns[1:]:
        agg_max_date = agg_max_date.withColumnRenamed(col, f"{col.lower()}_valor_max_date")


    # join the two datasets
    features = agg_mean.join(agg_max_date, on="prontuario", how="inner")

    return features

In [0]:

# construir meses de referencia a partir das datas minima e maxima
bounds = data.agg(
    F.date_trunc("month", F.min("dthr_alta_medica")).alias("min_m"),
    F.date_trunc("month", F.max("dthr_alta_medica")).alias("max_m"),
)

ref_dates_data = bounds.select(
    F.expr("sequence(min_m, max_m, interval 1 month) as ref_dates")
).select(F.explode("ref_dates").alias("ref_date"))

# lista de ref dates distintos
ref_dates = [r.ref_date for r in ref_dates_data.collect()]
len(ref_dates)

34

In [0]:
# define dataframe para incorporar dados ao cursor
features = spark.createDataFrame([], schema=StructType())

# para cada data de referencia, filtrar os dados anteriores a ela, para calculo das variaveis
for ref_ts in ref_dates:

    data_filtered = data.filter(
        (F.col("dthr_alta_medica") < F.lit(ref_ts)) &
        (F.col("dthr_alta_medica") >= F.add_months(F.lit(ref_ts), -12))
    )
    
    features_ref_date = generate_inter_features(data_filtered)

    features_ref_date = features_ref_date.withColumn("date_ref", F.lit(ref_ts).cast("timestamp"))

    features = features.unionByName(features_ref_date, allowMissingColumns=True)

    print(ref_ts)

2022-08-01 00:00:00
2022-09-01 00:00:00
2022-10-01 00:00:00
2022-11-01 00:00:00
2022-12-01 00:00:00
2023-01-01 00:00:00
2023-02-01 00:00:00
2023-03-01 00:00:00
2023-04-01 00:00:00
2023-05-01 00:00:00
2023-06-01 00:00:00
2023-07-01 00:00:00
2023-08-01 00:00:00
2023-09-01 00:00:00
2023-10-01 00:00:00
2023-11-01 00:00:00
2023-12-01 00:00:00
2024-01-01 00:00:00
2024-02-01 00:00:00
2024-03-01 00:00:00
2024-04-01 00:00:00
2024-05-01 00:00:00
2024-06-01 00:00:00
2024-07-01 00:00:00
2024-08-01 00:00:00
2024-09-01 00:00:00
2024-10-01 00:00:00
2024-11-01 00:00:00
2024-12-01 00:00:00
2025-01-01 00:00:00
2025-02-01 00:00:00
2025-03-01 00:00:00
2025-04-01 00:00:00
2025-05-01 00:00:00


In [0]:
data.printSchema()

root
 |-- atendimento: long (nullable = true)
 |-- cid: string (nullable = true)
 |-- prontuario: double (nullable = true)
 |-- dthr_internacao: timestamp (nullable = true)
 |-- dthr_alta_medica: timestamp (nullable = true)
 |-- dt_saida_paciente: string (nullable = true)
 |-- descricao_anamnese: string (nullable = true)
 |-- descricao_nota_adicional_anamnese: string (nullable = true)
 |-- descricao_origem_evento: string (nullable = true)
 |-- flag_obito_internacao: string (nullable = true)
 |-- dt_obito: string (nullable = true)
 |-- descricao_tipo_alta_medica: string (nullable = true)
 |-- dias_internado: integer (nullable = true)
 |-- cid_agg: string (nullable = true)
 |-- cid_agg_top50: string (nullable = true)



In [0]:
#display(data.select())

In [0]:
### Feature Engin